# 双液滴聚合：无涨落与 OU–SE

比较三个可检查的量：液桥高度代理、中心连线截面、俯视高度场。只读取真实共同时间步，不填补尚未保存或失败的数据。单条轨迹不是集合平均。

## 1. 配置与来源

`COALESCENCE_DATA_DIR` 指向包含 `det/` 和 `ou/` 的目录；两者需为同一双液滴几何和相同时间步。前驱膜始终存在，液桥指标为两滴中心之间、中心连线上高度的最小值，是可重复的一维代理，并非二维鞍点或分子首次接触。

In [ ]:
import os, re, struct, json, hashlib, shutil
from pathlib import Path
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

DATA = Path(os.environ.get('COALESCENCE_DATA_DIR', 'data')).resolve()
OUT = Path(os.environ.get('COALESCENCE_FIG_DIR', 'figures')).resolve()
OUT.mkdir(parents=True, exist_ok=True)
missing = [x for x in ('latex', 'dvipng', 'dvips') if shutil.which(x) is None]
if missing:
    raise RuntimeError('学术图需要 LaTeX、dvipng、dvips；缺少：' + ', '.join(missing))
mpl.rcParams.update({'text.usetex': True, 'text.latex.preamble': r'\usepackage{amsmath}',
 'figure.figsize': (8, 6), 'font.family':'serif', 'font.size':20,
 'axes.labelsize':20, 'axes.titlesize':20, 'xtick.labelsize':20, 'ytick.labelsize':20,
 'legend.fontsize':20, 'axes.grid':False, 'axes.spines.top':False, 'axes.spines.right':False,
 'xtick.direction':'out', 'ytick.direction':'out', 'savefig.dpi':600})
print('Input:', DATA, '\nFigures:', OUT)

## 2. 读取并验证原生叶单元

对静态 AMR 单元按面积保守展开至最细网格，不做平滑或插值。中心线 `y=32` 位于两行之间，使用两行均值。

In [ ]:
def load_case(name):
    folder = DATA / name
    config = json.loads((folder / 'runtime_config.json').read_text(encoding='utf-8'))
    activation = json.loads((folder / 'activation_config.json').read_text(encoding='utf-8'))
    assert activation['threshold'] == 0.03 and activation['permanent']
    initial = folder / 'initial.native'
    assert initial.is_file(), f'缺少 {initial}'
    fields = {0: initial}
    for p in folder.glob('field_step*.native'):
        m = re.fullmatch(r'field_step(\d+)\.native', p.name)
        if m:
            step = int(m.group(1)); assert step not in fields
            fields[step] = p
    return config, activation, fields

source_hashes = {}
def native(path):
    raw = path.read_bytes()
    source_hashes[str(path)] = hashlib.sha256(raw).hexdigest()
    cols, ncell, length = struct.unpack_from('<iQd', raw, 0)
    if cols != 6 or length != 64 or len(raw) != 20 + 48 * ncell:
        raise ValueError(f'文件未写完或格式不正确: {path}')
    leaves = np.frombuffer(raw, dtype='<f8', offset=20).reshape(ncell, 6)
    assert np.isfinite(leaves).all() and np.all(leaves[:,3] > 0)
    return leaves

def centre_section(leaves):
    out = np.full((2, 256), np.nan)
    for row, yy in enumerate((31.875, 32.125)):
        inside = leaves[(leaves[:,1] - leaves[:,2]/2 <= yy) & (yy < leaves[:,1] + leaves[:,2]/2)]
        for x, _, d, h, _, _ in inside:
            i = int(round((x - d/2) * 4)); width = int(round(d * 4))
            assert np.isnan(out[row, i:i+width]).all()
            out[row, i:i+width] = h
    assert np.isfinite(out).all()
    return out.mean(axis=0)

def fine_grid(leaves):
    grid = np.full((256,256), np.nan)
    for x, y, d, h, _, _ in leaves:
        i, j, k = int(round((x-d/2)*4)), int(round((y-d/2)*4)), int(round(d*4))
        assert np.isnan(grid[j:j+k, i:i+k]).all()
        grid[j:j+k, i:i+k] = h
    assert np.isfinite(grid).all()
    return grid

det, det_mask, det_fields = load_case('det')
ou, ou_mask, ou_fields = load_case('ou')
assert det['mode'] == -1 and ou['mode'] == 2 and det['dt'] == ou['dt']
dt = det['dt']
init_det, init_ou = native(det_fields[0]), native(ou_fields[0])
assert init_det.shape == init_ou.shape and np.array_equal(init_det[:,[0,1,2,5]],init_ou[:,[0,1,2,5]])
assert np.max(np.abs(init_det[:,3]-init_ou[:,3])) <= 1e-8, '两条轨迹的初场不同'
assert np.max(np.abs(init_det[:,4]-init_ou[:,4])) <= 1e-6, '两条轨迹的初始压力不同'
common = sorted(set(det_fields) & set(ou_fields))
assert common and common[0] == 0
x = (np.arange(256) + 0.5) * 0.25
print('Deterministic:', det, '\nOU--SE:', ou, '\nCommon frames:', len(common), 'last t:', common[-1]*dt)

## 3. 液桥高度代理

固定取两滴初始中心之间的中心线最小高度。两行平均只用于读取几何中心线，不是环向平均；随时间运行到更晚阶段时需重新检查该固定区间是否仍代表液桥。

In [ ]:
corridor = (x >= 27.25) & (x <= 36.75)
def bridge_trace(paths):
    return np.array([centre_section(native(paths[s]))[corridor].min() for s in common])
h_det, h_ou = bridge_trace(det_fields), bridge_trace(ou_fields)
fig, ax = plt.subplots()
ax.plot(np.array(common)*dt, h_det, color='black', label='Deterministic')
ax.plot(np.array(common)*dt, h_ou, color='#D62728', ls='--', label=r'OU--SE')
ax.set(xlabel=r'Time $t$', ylabel=r'Centerline bridge height $h_b$')
ax.yaxis.set_major_formatter(mpl.ticker.FormatStrFormatter('%.8f'))
ax.legend(loc='lower center', bbox_to_anchor=(.5,1.02), ncol=2, frameon=False)
ax.grid(False, which='both'); fig.subplots_adjust(left=.23,right=.97,bottom=.14,top=.78)
fig.savefig(OUT/'01_bridge_height.pdf'); fig.savefig(OUT/'01_bridge_height.png', dpi=600)
plt.show()

## 4. 中心连线截面

默认显示两条轨迹实际存在的最晚共同帧；可设置环境变量 `COALESCENCE_PLOT_TIME` 指定同样已保存的时间。

In [ ]:
requested = os.environ.get('COALESCENCE_PLOT_TIME')
selected = common[-1] if requested is None else int(round(float(requested)/dt))
if selected not in common or (requested is not None and not np.isclose(selected*dt, float(requested), atol=1e-12)):
    raise ValueError('要求的物理时刻不是两条轨迹的共同已存帧')
t = selected * dt
a, b = native(det_fields[selected]), native(ou_fields[selected])
fig, ax = plt.subplots()
ax.plot(x, centre_section(a), color='black', label='Deterministic')
ax.plot(x, centre_section(b), color='#D62728', ls='--', label='OU--SE')
ax.set(xlim=(21,43), xlabel=r'$x$', ylabel=r'Height $h$')
fig.text(.5,.95,rf'$t={t:g}$',ha='center',va='center')
ax.legend(loc='lower center',bbox_to_anchor=(.5,1.02),ncol=2,frameon=False)
ax.grid(False,which='both');fig.subplots_adjust(left=.16,right=.97,bottom=.14,top=.77)
fig.savefig(OUT/'02_center_section.pdf'); fig.savefig(OUT/'02_center_section.png', dpi=600)
plt.show()

## 5. 俯视图

两幅图使用同一时刻和同一高度色标；虚线为中心连线。高度阈值与历史激活状态不是同一个量。

In [ ]:
da, db = fine_grid(a), fine_grid(b)
vmin, vmax = min(da.min(),db.min()), max(da.max(),db.max())
fig, axes = plt.subplots(1,2,figsize=(8,6),sharex=True,sharey=True)
for ax, field, label in zip(axes, (da,db), ('Deterministic','OU--SE')):
    im = ax.imshow(field, origin='lower', extent=(0,64,0,64), cmap='viridis',
                   interpolation='nearest', vmin=vmin, vmax=vmax)
    ax.axhline(32,color='white',ls='--',lw=1)
    ax.set(xlim=(21,43), ylim=(24,40), xlabel=r'$x$', title=label)
    ax.grid(False,which='both')
axes[0].set_ylabel(r'$y$')
fig.suptitle(rf'$t={t:g}$')
fig.subplots_adjust(left=.12,right=.80,bottom=.2,top=.78,wspace=.1)
cax=fig.add_axes([.82,.26,.025,.42]);fig.colorbar(im,cax=cax,label=r'Height $h$')
fig.savefig(OUT/'03_top_view.pdf');fig.savefig(OUT/'03_top_view.png',dpi=600)
plt.show()
print('Selected source SHA256:')
for name,path in [('det',det_fields[selected]),('ou',ou_fields[selected])]:
    print(name, selected, hashlib.sha256(path.read_bytes()).hexdigest(), path)
(OUT/'source_hashes.json').write_text(json.dumps(source_hashes,indent=2),encoding='utf-8')

## 6. 使用边界

高度门控只限制直接随机源，不能保证涨落–耗散平衡。不能从一个 seed 或一段短轨迹推断聚合位置正态分布、统计收敛、渐近速度或分子尺度接触。